In [ ]:
using Random
using Statistics
using Printf
using LinearAlgebra
using Plots
using Logging

function find_project_root(start::AbstractString=pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "System1D.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate project root from $start")
        dir = parent
    end
end

PROJECT_ROOT = find_project_root()

include(joinpath(PROJECT_ROOT, "Experiments", "common", "notebook_helpers.jl"))

NOTEBOOK_REL_DIR = joinpath("Experiments", "systems", "hardcore_boson_ring_1d", "gfmc", "notebooks")
PATHS = nb_paths(PROJECT_ROOT, NOTEBOOK_REL_DIR)
nb_include_formatting(PATHS.notebook_dir)

include(joinpath(PROJECT_ROOT, "src", "System1D.jl"))
using .System1D

default(; dpi=170)
nothing


## Model and GFMC Parameters

This notebook runs guided fixed-population GFMC for `N` hard-core bosons on a periodic ring with a cosine lattice and pairwise Coulomb repulsion.

The Hamiltonian is built on the generic `Hamiltonian` path:
`H = -D sum_i d^2/dx_i^2 + sum_i V0 * cos(2*pi*x_i/a) + sum_{i<j} V_pair(r_ij)`
where `r_ij` is the periodic minimum-image separation.

The pair interaction contains two pieces:
- a very large short-range barrier for `r_ij < r_core` to model hard-core exclusion
- a softened Coulomb repulsion `g_coul / sqrt(r_ij^2 + coulomb_softening^2)`

The guiding state is a positive bosonic Jastrow ansatz with:
- a one-body lattice factor `exp(lambda * sum_i cos(2*pi*x_i/a))`
- a pair factor `prod_{i<j} |sin(pi * (x_i - x_j) / L)|^pair_strength`

Parameters used below:
- Boson count `N = 3`
- Number of lattice periods `M = 4`
- Lattice spacing `a = 1.0`
- Ring length `L = 4.0`
- Lattice amplitude `V0 = -1.5`
- Coulomb strength `g_coul = 0.8`
- Coulomb softening `coulomb_softening = 0.05 * a`
- Hard-core radius `r_core = 0.18 * a`
- Hard-core barrier `hard_core_barrier = 1000.0`
- Time step `dt = 1.0e-3`
- Total steps `nsteps = 1200`
- Equilibration steps `nequil = 200`
- Target population `targetN = 600`

Trial / node structure:
- Guiding policy: `ImportanceGuiding(trial, H)`
- Node policy: `NoNode()`
- Reconfiguration policy: `SystematicReconfiguration()`


## Julia Construction

The next cell defines the periodic many-boson Hamiltonian, the positive Jastrow trial state, the hard-core initialization helper, and the GFMC runtime controls.

Edit that cell if you want to change the boson number, interaction strength, hard-core radius, or density-smoothing settings.


In [ ]:
N = 3
M = 4
a = 1.0
L = M * a
V0 = -1.5
D = 0.5

g_coul = 0.8
coulomb_softening = 0.05 * a
r_core = 0.18 * a
hard_core_barrier = 1000.0

lambda_trial = -0.5 * V0
pair_strength = 0.75
pair_eps = 1.0e-8

bc = PeriodicBoundary1D(0.0, L)
k_lat = 2pi / a
pair_alpha = pi / L

pair_distance(xi, xj) = abs(displacement(bc, xj, xi))

function pair_potential(r)
    rep = g_coul / sqrt(r^2 + coulomb_softening^2)
    return r < r_core ? (hard_core_barrier + rep) : rep
end

function V(R)
    total = 0.0
    @inbounds for i in 1:N
        total += V0 * cos(k_lat * R[i])
    end
    @inbounds for i in 1:(N - 1)
        for j in (i + 1):N
            total += pair_potential(pair_distance(R[i], R[j]))
        end
    end
    return total
end

H = Hamiltonian(N, D, V, bc)

function boson_trial_terms(R)
    logabs = 0.0
    grad = zeros(Float64, N)
    lapl = 0.0

    @inbounds for i in 1:N
        xi = wrap_coordinate(bc, R[i])
        coskx = cos(k_lat * xi)
        sinkx = sin(k_lat * xi)
        logabs += lambda_trial * coskx
        grad[i] = -lambda_trial * k_lat * sinkx
        lapl += -lambda_trial * k_lat^2 * coskx
    end

    @inbounds for i in 1:(N - 1)
        for j in (i + 1):N
            dx_ij = displacement(bc, R[j], R[i])
            u = pair_alpha * dx_ij
            s = sin(u)
            s_abs = abs(s)
            s_eff = s_abs < pair_eps ? pair_eps : s_abs
            cot_u = cos(u) / (s >= 0 ? s_eff : -s_eff)

            logabs += pair_strength * log(s_eff)
            c = pair_strength * pair_alpha * cot_u
            grad[i] += c
            grad[j] -= c
            lapl += -2.0 * pair_strength * pair_alpha^2 / (s_eff * s_eff)
        end
    end

    return logabs, grad, lapl
end

logpsi(R) = first(boson_trial_terms(R))
gradlogpsi(R) = boson_trial_terms(R)[2]
lapllogpsi(R) = boson_trial_terms(R)[3]
signpsi(R) = 1.0

trial = TrialWF(logpsi, gradlogpsi, lapllogpsi, signpsi)
guiding = ImportanceGuiding(trial, H)

function valid_hardcore_configuration(R)
    @inbounds for i in 1:(length(R) - 1)
        for j in (i + 1):length(R)
            pair_distance(R[i], R[j]) > r_core || return false
        end
    end
    return true
end

function sample_hardcore_ring_configurations(nwalkers::Integer, rng::AbstractRNG; max_tries::Integer=10000)
    nwalkers_int = Int(nwalkers)
    X = Matrix{Float64}(undef, N, nwalkers_int)
    @inbounds for w in 1:nwalkers_int
        accepted = false
        for _ in 1:max_tries
            R = rand(rng, N) .* L
            if valid_hardcore_configuration(R)
                X[:, w] = R
                accepted = true
                break
            end
        end
        accepted || error("Failed to sample a valid hard-core configuration after $(max_tries) tries")
    end
    return X
end

targetN = 600
dt = 1.0e-3
nsteps = 1200
nequil = 200
ET0 = 0.0
feedback = 0.1
reconfiguration_interval = 5
branch_cap = 5.0
energy_window = 40

params = GFMCParams(dt, nsteps, nequil, targetN, ET0, feedback, reconfiguration_interval, branch_cap, energy_window)
RECONFIGURATION = SystematicReconfiguration()

rng_init = MersenneTwister(1234)
initial_positions = sample_hardcore_ring_configurations(targetN, rng_init)

SNAPSHOT_STEPS = nb_default_snapshot_steps(nsteps)
DENSITY_GRID_POINTS = 400
DENSITY_BANDWIDTH = 0.12 * a
PAIR_SEP_BINS = 120
PERIOD_MARKERS = collect(0.0:a:L)

RUN_LABEL = "guided hard-core bosons"
RUN_COLOR = :teal
PLOT_TITLE = "Hard-core boson ring GFMC"
DENSITY_TITLE = "Hard-core boson ring GFMC: pooled one-body densities"
UNPOOLED_DENSITY_TITLE = "Hard-core boson ring GFMC: final per-particle one-body densities"
PAIR_TITLE = "Hard-core boson ring GFMC: final pair-separation density"
PARTICLE_COLORS = [:teal, :darkorange, :navy, :crimson, :purple, :goldenrod, :deeppink, :forestgreen]

SHOW_PROGRESS = false
PROGRESS_EVERY = 0
DEBUG_MODE = false
DEBUG_EVERY = 20
WRITE_RUN_CSV = false
CSV_FILENAME = "hardcore_boson_ring_gfmc.csv"
SAVE_FIGURES = false
FIGURE_STEM = "hardcore_boson_ring_gfmc"


In [ ]:
sim = GFMCSim(
    H,
    params,
    initial_positions,
    MersenneTwister(52);
    guiding=guiding,
    nodepolicy=NoNode(),
    reconfiguration=RECONFIGURATION,
)
run_gfmc!(
    sim;
    snapshot_steps=SNAPSHOT_STEPS,
    show_progress=SHOW_PROGRESS,
    progress_every=PROGRESS_EVERY,
    progress_label=RUN_LABEL,
    debug=DEBUG_MODE,
    debug_every=DEBUG_EVERY,
)

start_idx = min(params.nequil + 1, length(sim.energy_mean_history))
mean_energy, sem_energy = nb_mean_sem(sim.energy_mean_history[start_idx:end])
final_snapshot = nb_last_snapshot(sim)
final_pair_sep = Float64[]
for R in final_snapshot
    for i in 1:(length(R) - 1)
        for j in (i + 1):length(R)
            push!(final_pair_sep, pair_distance(R[i], R[j]))
        end
    end
end

println(@sprintf("%s mean energy after nequil=%d: %.8f +/- %.3e", RUN_LABEL, params.nequil, mean_energy, sem_energy))
println("final fixed walker count = ", sim.population_history[end])
println(@sprintf("final mean weight = %.6f", sim.mean_weight_history[end]))
println(@sprintf("final effective population = %.2f", sim.effective_population_history[end]))
println(@sprintf("minimum final pair separation = %.6f", minimum(final_pair_sep)))
println("count with r < r_core = ", count(r -> r < r_core, final_pair_sep))

if WRITE_RUN_CSV
    csv_path = joinpath(PATHS.tables_dir, CSV_FILENAME)
    nb_write_csv(csv_path, nb_gfmc_rows(RUN_LABEL, sim))
    println("Wrote run CSV to: ", abspath(csv_path))
end


In [ ]:
function pooled_ring_coordinates(snapshot)
    xs = Float64[]
    for R in snapshot
        append!(xs, Float64.(R))
    end
    return xs
end

function ring_particle_coordinates(snapshot, particle_idx::Integer)
    idx = Int(particle_idx)
    return Float64[R[idx] for R in snapshot]
end

function pair_separations(snapshot)
    rs = Float64[]
    for R in snapshot
        for i in 1:(length(R) - 1)
            for j in (i + 1):length(R)
                push!(rs, pair_distance(R[i], R[j]))
            end
        end
    end
    return rs
end

history_fig = nb_plot_gfmc_history([sim]; labels=[RUN_LABEL], colors=[RUN_COLOR], title_prefix=PLOT_TITLE)
display(history_fig)
nb_save_figure(history_fig, PATHS.figures_dir, FIGURE_STEM, "history"; enabled=SAVE_FIGURES)

available_steps = SNAPSHOT_STEPS[1:min(length(SNAPSHOT_STEPS), length(sim.walker_positions_history))]
density_fig = plot(
    xlabel="x",
    ylabel="pooled one-body density",
    title=DENSITY_TITLE,
    legend=:topright,
    xlims=(0.0, L),
)
for (snapshot, step_idx) in zip(sim.walker_positions_history, available_steps)
    xs = pooled_ring_coordinates(snapshot)
    centers, density = nb_periodic_kde_curve(
        xs;
        xmin=0.0,
        xmax=L,
        grid_points=DENSITY_GRID_POINTS,
        bandwidth=DENSITY_BANDWIDTH,
    )
    plot!(density_fig, centers, density; label="step $(step_idx)", color=RUN_COLOR, linewidth=2.2, alpha=0.82)
end
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(density_fig, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "cell markers" : ""))
end

display(density_fig)
nb_save_figure(density_fig, PATHS.figures_dir, FIGURE_STEM, "density"; enabled=SAVE_FIGURES)

final_snapshot = nb_last_snapshot(sim)
particle_colors = [PARTICLE_COLORS[1 + mod(i - 1, length(PARTICLE_COLORS))] for i in 1:N]

unpooled_density_fig = plot(
    xlabel="x",
    ylabel="one-body density",
    title=UNPOOLED_DENSITY_TITLE,
    legend=:topright,
    xlims=(0.0, L),
)
for particle_idx in 1:N
    xs = ring_particle_coordinates(final_snapshot, particle_idx)
    centers, density = nb_periodic_kde_curve(
        xs;
        xmin=0.0,
        xmax=L,
        grid_points=DENSITY_GRID_POINTS,
        bandwidth=DENSITY_BANDWIDTH,
    )
    plot!(unpooled_density_fig, centers, density; label="particle $(particle_idx)", color=particle_colors[particle_idx], linewidth=2.4)
end
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(unpooled_density_fig, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "cell markers" : ""))
end

display(unpooled_density_fig)
nb_save_figure(unpooled_density_fig, PATHS.figures_dir, FIGURE_STEM, "density_unpooled"; enabled=SAVE_FIGURES)

final_pair_sep = pair_separations(nb_last_snapshot(sim))
centers, density = nb_density_curve(
    final_pair_sep;
    nbins=PAIR_SEP_BINS,
    xmin=0.0,
    xmax=0.5 * L,
    smoothing_window=9,
)
pair_fig = plot(
    centers,
    density;
    xlabel="r",
    ylabel="density",
    title=PAIR_TITLE,
    label="final pair-separation density",
    color=:darkorange,
    linewidth=2.4,
    xlims=(0.0, 0.5 * L),
)
vline!(pair_fig, [r_core]; color=:black, linestyle=:dash, linewidth=1.2, label="r_core")

display(pair_fig)
nb_save_figure(pair_fig, PATHS.figures_dir, FIGURE_STEM, "pair_sep"; enabled=SAVE_FIGURES)
